In [1]:
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import shapely
import sqlalchemy

In [2]:
data_path = Path(os.environ["DATA_PATH"])
generated_path = data_path / "generated"
population_grids_path = Path(os.environ["POPULATION_GRIDS_PATH"])
denue_path = Path(os.environ["DENUE_PATH"])

In [3]:
DENUE_TO_AMENITY_MAPPING = [
    # Salud
    {
        "name": "Consultorios médicos",
        "query": 'codigo_act.str.match("^621")',
    },
    {
        "name": "Hospital general",
        "query": 'codigo_act.str.match("^622")',
    },
    {
        "name": "Farmacia",
        "query": 'codigo_act.str.match("^46411")',
    },
    # Recreativo
    {
        "name": "Cine",
        "query": 'codigo_act.str.match("^51213")',
    },
    {
        "name": "Otros servicios recreativos",
        "query": 'codigo_act.str.match("^(71399|712|713)")',
    },
    {
        "name": "Clubs deportivos y de acondicionamiento físico",
        "query": 'codigo_act.str.match("^(71391|71394)")',
    },
    # Educación
    {
        "name": "Guarderia",
        "query": 'codigo_act.str.match("^6244")',
    },
    {
        "name": "Educación preescolar",
        "query": 'codigo_act.str.match("^61111")',
    },
    {"name": "Educación primaria", "query": 'codigo_act.str.match("^61112")'},
    {"name": "Educación secundaria", "query": 'codigo_act.str.match("^(61113|61114)")'},
    {
        "name": "Educación media superior",
        "query": 'codigo_act.str.match("^(61115|61116)")',
    },
    {"name": "Educación superior", "query": 'codigo_act.str.match("^(6112|6113)")'},
]

In [4]:
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}",
)

In [5]:
df_agebs = gpd.read_file(generated_path / "agebs.gpkg")

In [6]:
bbox = shapely.box(*df_agebs.total_bounds).buffer(
    1000,
    cap_style="flat",
    join_style="mitre",
)

In [7]:
df_mesh = gpd.read_file(
    population_grids_path / "initial" / "mesh" / "nivel9_1.shp",
).to_crs("EPSG:6372")
df_mesh = df_mesh[df_mesh.intersects(bbox)].copy()

mesh_agg = (
    df_mesh.overlay(
        df_agebs.drop(columns=["zone"]).assign(ageb_area=lambda df: df.area),
    )
    .assign(
        area_fraction=lambda df: df.area / df.ageb_area,
    )
    .drop(columns=["ageb_area", "VIVPAR_HAB", "TVIVPAR", "geometry"], errors="ignore")
)

for c in mesh_agg.columns:
    if not c.startswith("P"):
        continue
    mesh_agg[c] = mesh_agg[c] * mesh_agg["area_fraction"]
mesh_agg = mesh_agg.drop(columns="area_fraction").groupby("CODIGO").sum()

df_mesh = df_mesh.merge(mesh_agg, on="CODIGO", how="left").fillna(0.0)

# Amenities

## DENUE

In [8]:
df_denue: list[gpd.GeoDataFrame] = []

for path in (denue_path / "05-2025").glob("*"):
    if not path.is_dir():
        continue

    shp_list = list((path / "conjunto_de_datos").glob("*.shp"))
    if len(shp_list) == 0:
        err = f"No shapefile found in {path}"
        raise FileNotFoundError(err)

    if len(shp_list) > 1:
        err = f"Multiple shapefiles found in {path}"
        raise FileExistsError(err)

    temp = gpd.read_file(shp_list[0], columns=["per_ocu", "codigo_act", "id"]).to_crs(
        "EPSG:6372",
    )
    temp = temp[temp.intersects(bbox)]
    df_denue.append(temp)

df_denue = (
    pd.concat(df_denue, ignore_index=True)
    .drop_duplicates(subset=["id"])
    .drop(columns=["id"])
    .pipe(gpd.GeoDataFrame, crs="EPSG:6372", geometry="geometry")
    .assign(
        num_workers=lambda x: x["per_ocu"].map(
            {
                "0 a 5 personas": 3,
                "6 a 10 personas": 8,
                "11 a 30 personas": 20,
                "31 a 50 personas": 40,
                "51 a 100 personas": 75,
                "101 a 250 personas": 175,
                "251 y más personas": 500,
            },
        ),
        codigo_act=lambda x: x["codigo_act"].astype(str),
    )[["geometry", "codigo_act", "num_workers"]]
)

for elem in DENUE_TO_AMENITY_MAPPING:
    idx = df_denue.query(elem["query"]).index
    df_denue.loc[idx, "amenity"] = elem["name"]

df_denue = df_denue.dropna(subset=["amenity"]).drop(columns=["codigo_act"])

## Parques

In [ ]:
df_park = gpd.read_file(data_path / "initial")